In [1]:
import os
import cv2
import numpy as np
from PIL import Image 

In [19]:
import cv2
import os

def create_user(f_id, name):
    # 1. Try a different backend (CAP_DSHOW) if the camera is stuck
    web = cv2.VideoCapture(0, cv2.CAP_DSHOW) 
    web.set(3, 640)
    web.set(4, 480)

    xml_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    faces = cv2.CascadeClassifier(xml_path)

    # 2. Check if classifier loaded
    if faces.empty():
        print("Error: Could not load XML classifier.")
        return

    f_dir = "dataset"
    folder_path = os.path.join(f_dir, name)
    os.makedirs(folder_path, exist_ok=True)

    print(f"Starting collection for {name}. Look at the camera...")
    counter = 0

    try:
        while True:
            res, frame = web.read()
            
            # 3. Warm-up check: If camera isn't ready yet, keep trying
            if not res:
                continue
                
            img = cv2.flip(frame, 1)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            multi_face = faces.detectMultiScale(gray, 1.3, 5)
            
            for (x, y, w, h) in multi_face:
                counter += 1
                cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)

                file_name = f"{name}.{f_id}.{counter}.jpg"
                full_save_path = os.path.join(folder_path, file_name)
                
                # Save the grayscale face crop
                cv2.imwrite(full_save_path, gray[y:y+h, x:x+w])
            
            # Show the image even if no face is detected so the window stays open
            cv2.imshow("Capture - Press 'q' to Quit", img)

            # 4. Break conditions
            if cv2.waitKey(1) & 0xff == ord("q"):
                print("Interrupted by user.")
                break
            if counter >= 40:
                print("Successfully collected 40 samples.")
                break

    except Exception as e:
        print(f"An error occurred: {e}")

    finally:
        # 5. This block ensures the camera turns OFF no matter what
        web.release()
        cv2.destroyAllWindows()
        print("Camera released and windows closed.")

# create_user(1, "kamran")

In [20]:
create_user(1,"kamran")

Starting collection for kamran. Look at the camera...
Interrupted by user.
Camera released and windows closed.


In [2]:
def train():
    database = "dataset"
    # Ensure the database exists to avoid errors
    if not os.path.exists(database):
        print(f"Error: Folder '{database}' not found.")
        return

    # Filter out the generator issue by casting to list
    img_dir = [x[0] for x in list(os.walk(database))[1:]]
    
    # Initialize the recognizer and detector
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    xml_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    detector = cv2.CascadeClassifier(xml_path)
    
    face_samples = []
    ids = []

    for path in img_dir:
        # Filter for images only to avoid crashing on hidden system files
        img_paths = [os.path.join(path, f) for f in os.listdir(path) if f.endswith(('.jpg', '.png', '.jpeg'))]

        for img_path in img_paths:
            try:
                # Open image and convert to grayscale
                PIL_img = Image.open(img_path).convert('L')
                img_numpy = np.array(PIL_img, "uint8")
                
                # Extract ID from filename: "name.id.number.jpg" -> index 1 is id
                file_name = os.path.split(img_path)[-1]
                id = int(file_name.split(".")[1])
                
                # Detect face in the saved training image
                faces = detector.detectMultiScale(img_numpy)

                for (x, y, w, h) in faces:
                    # CRITICAL: Crop and resize so all samples are identical size
                    face_crop = img_numpy[y:y+h, x:x+w]
                    face_resized = cv2.resize(face_crop, (100, 100))
                    
                    face_samples.append(face_resized)
                    ids.append(id)
            except Exception as e:
                print(f"Skipping file {img_path} due to error: {e}")

    if len(face_samples) == 0:
        print("No training data found. Check your dataset folder.")
        return

    # Train and save
    recognizer.train(face_samples, np.array(ids))
    recognizer.write("trainer.yml")
    print(f"Training complete. {len(np.unique(ids))} users trained.")

In [3]:
train()

AttributeError: module 'cv2' has no attribute 'face'